# KTTS speaker-matched HiFi-GAN v1
Fine-tunes a universal 22.05 kHz HiFi-GAN generator on KTTS. Non-commercial CC BY-NC 3.0 experiment.


In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher
!git clone -q --depth 1 https://github.com/jik876/hifi-gan.git /kaggle/working/hifi-gan
%cd /kaggle/working/hifi-gan
!pip install -q 'numpy<2' librosa soundfile scipy pyarrow huggingface_hub tensorboard
!python /kaggle/working/cipher/scripts/patch-hifigan-generator-warmstart.py /kaggle/working/hifi-gan/train.py
!python /kaggle/working/cipher/scripts/prepare-ktts-matcha.py --repo-dir /kaggle/working/hifi-gan --output /kaggle/working/ktts-vocoder --local-root /kaggle/input


In [ ]:
from pathlib import Path
import json
config = {
 'resblock':'1','num_gpus':1,'batch_size':16,'learning_rate':0.0001,'adam_b1':0.8,'adam_b2':0.99,'lr_decay':0.9995,'seed':1234,
 'upsample_rates':[8,8,2,2],'upsample_kernel_sizes':[16,16,4,4],'upsample_initial_channel':512,
 'resblock_kernel_sizes':[3,7,11],'resblock_dilation_sizes':[[1,3,5],[1,3,5],[1,3,5]],
 'segment_size':8192,'num_mels':80,'num_freq':1025,'n_fft':1024,'hop_size':256,'win_size':1024,
 'sampling_rate':22050,'fmin':0,'fmax':8000,'fmax_for_loss':None,'num_workers':2,
 'dist_config':{'dist_backend':'nccl','dist_url':'tcp://localhost:54321','world_size':1}
}
Path('/kaggle/working/hifi-gan/config_ktts.json').write_text(json.dumps(config, indent=2))
Path('/kaggle/working/hifi-gan/cp_hifigan').mkdir(exist_ok=True)
!wget -q https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000 -O /kaggle/working/hifi-gan/cp_hifigan/g_00000000
!python -c "import torch; p=torch.load('/kaggle/working/hifi-gan/cp_hifigan/g_00000000',map_location='cpu',weights_only=False); print('Universal generator tensors:',len(p['generator']))"


In [ ]:
import os, subprocess, sys
from pathlib import Path
env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'; env['PYTHONUNBUFFERED']='1'
log=Path('/kaggle/working/hifigan-training.log')
cmd=[sys.executable,'-u','train.py','--config','config_ktts.json','--input_wavs_dir','/','--input_training_file','/kaggle/working/ktts-vocoder/train.txt','--input_validation_file','/kaggle/working/ktts-vocoder/val.txt','--checkpoint_path','/kaggle/working/hifi-gan/cp_hifigan','--training_epochs','50','--stdout_interval','25','--checkpoint_interval','1000','--summary_interval','200','--validation_interval','1000']
with log.open('w') as handle: result=subprocess.run(cmd,cwd='/kaggle/working/hifi-gan',env=env,stdout=handle,stderr=subprocess.STDOUT,text=True,timeout=41400)
print('Training exit code:',result.returncode); print(log.read_text(errors='replace')[-16000:])
if result.returncode != 0: raise RuntimeError('HiFi-GAN training failed; inspect hifigan-training.log')


In [ ]:
from pathlib import Path
import json, shutil
root=Path('/kaggle/working/hifi-gan/cp_hifigan')
generators=sorted(root.glob('g_*'), key=lambda p:p.stat().st_mtime)
trained=[p for p in generators if p.name != 'g_00000000']
assert trained, f'No trained generator checkpoint found: {generators}'
release=Path('/kaggle/working/ktts-hifigan-v1'); release.mkdir(exist_ok=True)
shutil.copy2(trained[-1], release/'generator.pt')
shutil.copy2('/kaggle/working/hifi-gan/config_ktts.json', release/'config.json')
(release/'LICENSE-DATA.txt').write_text('KTTS data: CC BY-NC 3.0. Non-commercial use only. DOI: 10.17632/5c4dcvxdmb.2\n')
report={'epochs':50,'latestCheckpoint':trained[-1].name,'generatorWarmStart':'hifigan_univ_v1','speaker':422,'productionDeployed':False,'humanReviewRequired':True}
(release/'training-report.json').write_text(json.dumps(report,indent=2))
shutil.make_archive('/kaggle/working/ktts-hifigan-v1','zip',release)
print(report); print('Artifact: /kaggle/working/ktts-hifigan-v1.zip')
